# Procena broja stanovnika iz otisaka zgrada (pristup 2)

ResNet-18 (ImageNet) fine-tuning na rasterizovane otiske zgrada po naselju.
Ulaz su 2 kanala: pokrivenost (udeo celije pod zgradom) i zapreminska gustina (pokrivenost x spratnost).
Cilj je `log1p(broj_stanovnika)`. Podela po opstinama (GroupKFold). Eksperimenti kroz MLflow.

## Instalacija

In [ ]:
%pip install -q timm mlflow
try:
    dbutils.library.restartPython()
except NameError:
    pass

## Konfiguracija

In [ ]:
import os, glob, zipfile, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm, mlflow
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Colab: "/content/fp_data" (raspakuje footprint_upload.zip). Databricks: postavi na svoj Volume.
DATA_DIR = "/content/fp_data"
if os.path.isdir("/content") and not os.path.isdir(DATA_DIR + "/footprint_cutouts"):
    zipfile.ZipFile("/content/footprint_upload.zip").extractall(DATA_DIR)
CUT = DATA_DIR + "/footprint_cutouts"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
NB, PX = 2, 224                       # 2 kanala: pokrivenost, zapreminska gustina
EP_HEAD, EP_FT, BS = 3, 50, 64
LR_HEAD, LR_FT = 1e-3, 3e-4
NUM_WORKERS = 8
print("device:", DEVICE, "| footprint cutouts:", len(os.listdir(CUT)))

## Podaci i podela

In [ ]:
labele = pd.read_parquet(DATA_DIR + "/naselje_table.parquet")[
    ["naselje_maticni_broj", "opstina_maticni_broj", "pop"]]
df = pd.DataFrame({"path": glob.glob(CUT + "/*.npy")})
df["naselje_maticni_broj"] = df.path.map(lambda f: int(os.path.splitext(os.path.basename(f))[0]))
df = df.merge(labele, on="naselje_maticni_broj", how="inner")
df["y"] = np.log1p(df["pop"]).astype("float32")

splitter = GroupKFold(n_splits=min(5, df.opstina_maticni_broj.nunique()))
tr, va = next(splitter.split(df, groups=df.opstina_maticni_broj))
train_df = df.iloc[tr].reset_index(drop=True)
val_df = df.iloc[va].reset_index(drop=True)
print(f"uzoraka {len(df)} | opstina {df.opstina_maticni_broj.nunique()} | trening {len(train_df)} | validacija {len(val_df)}")

## Normalizacija i dataset

In [ ]:
uzorak = np.stack([np.load(p) for p in train_df.path.sample(min(400, len(train_df)), random_state=SEED)])
MEAN = uzorak.mean((0, 2, 3), keepdims=True).astype("float32")
STD = uzorak.std((0, 2, 3), keepdims=True).astype("float32") + 1e-6

class Naselja(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        red = self.frame.iloc[i]
        x = (np.load(red.path).astype("float32") - MEAN[0]) / STD[0]
        if self.augment:
            if np.random.rand() < 0.5: x = x[:, :, ::-1]
            if np.random.rand() < 0.5: x = x[:, ::-1, :]
            x = np.rot90(x, np.random.randint(4), axes=(1, 2))
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor([red.y], dtype=torch.float32)

def seed_worker(wid):
    s = SEED + wid
    np.random.seed(s); random.seed(s)

gen = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(Naselja(train_df, augment=True), batch_size=BS, shuffle=True, generator=gen,
                      num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=4,
                      persistent_workers=True, worker_init_fn=seed_worker)
val_dl = DataLoader(Naselja(val_df), batch_size=BS, num_workers=NUM_WORKERS,
                    pin_memory=True, persistent_workers=True)

## Model

In [ ]:
# timm prilagodjava prvi konvolucioni sloj na 2 kanala (in_chans)
model = timm.create_model("resnet18", pretrained=True, in_chans=NB, num_classes=1).to(DEVICE)
loss_fn = nn.HuberLoss()
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def prodji(loader, treniraj, optim=None, freeze_bn=False):
    model.train(treniraj)
    if freeze_bn:
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()
    ukupno, P, Y = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(treniraj), torch.autocast("cuda", enabled=USE_AMP):
            out = model(x)
            loss = loss_fn(out, y)
        if treniraj:
            optim.zero_grad()
            scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
        ukupno += loss.item() * len(x)
        P.append(out.detach().float().cpu().numpy()); Y.append(y.cpu().numpy())
    return ukupno / len(loader.dataset), np.concatenate(P).ravel(), np.concatenate(Y).ravel()

## Trening (sa MLflow pracenjem)

In [ ]:
mlflow.start_run()
mlflow.log_params({"pristup": "footprint", "backbone": "resnet18", "kanali": NB, "px": PX, "batch": BS,
                   "ep_head": EP_HEAD, "ep_ft": EP_FT, "lr_head": LR_HEAD, "lr_ft": LR_FT,
                   "seed": SEED, "n_uzoraka": len(df), "amp": USE_AMP})
istorija = []
best_r2, best_state = -1e9, None

for naziv, param in model.named_parameters():
    param.requires_grad = naziv.startswith("fc")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)
for e in range(EP_HEAD):
    tl, _, _ = prodji(train_dl, True, opt, freeze_bn=True)
    vl, P, Y = prodji(val_dl, False)
    r2 = r2_score(Y, P); istorija.append((tl, vl, r2))
    mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=e)
    print(f"[glava {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

for param in model.parameters():
    param.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=LR_FT)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP_FT)
for e in range(EP_FT):
    tl, _, _ = prodji(train_dl, True, opt); sched.step()
    vl, P, Y = prodji(val_dl, False)
    r2 = r2_score(Y, P); istorija.append((tl, vl, r2))
    mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=EP_HEAD + e)
    if r2 > best_r2:
        best_r2 = r2
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"[fine {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

model.load_state_dict(best_state)
mlflow.log_metric("best_val_r2", best_r2)
print(f"najbolji validacioni R2: {best_r2:.3f}")

## Evaluacija

In [ ]:
_, P, Y = prodji(val_dl, False)
pred, stvarno = np.expm1(P), np.expm1(Y)
metrike = {"val_r2_log": r2_score(Y, P),
           "val_mae_log": mean_absolute_error(Y, P),
           "val_rmse_log": mean_squared_error(Y, P) ** 0.5,
           "val_mae_pop": mean_absolute_error(stvarno, pred),
           "val_rmse_pop": mean_squared_error(stvarno, pred) ** 0.5}

val_df2 = val_df.assign(pred=pred, stvarno=stvarno)
po_opstini = val_df2.groupby("opstina_maticni_broj")[["pred", "stvarno"]].sum()
metrike["agg_opstina_r2"] = r2_score(po_opstini.stvarno, po_opstini.pred) if len(po_opstini) > 1 else float("nan")
mlflow.log_metrics({k: float(v) for k, v in metrike.items()})
for k, v in metrike.items():
    print(f"{k}: {v:.3f}")

ep = range(len(istorija))
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax[0, 0].plot(ep, [h[0] for h in istorija], label="trening")
ax[0, 0].plot(ep, [h[1] for h in istorija], label="validacija")
ax[0, 0].axvline(EP_HEAD - 0.5, ls=":", color="gray")
ax[0, 0].set_title("Huber gubitak po epohi"); ax[0, 0].set_xlabel("epoha"); ax[0, 0].legend()
ax[0, 1].plot(ep, [h[2] for h in istorija]); ax[0, 1].axhline(0, color="red", ls="--")
ax[0, 1].set_title("Validacioni R2 po epohi"); ax[0, 1].set_xlabel("epoha")
m = max(stvarno.max(), pred.max(), 1)
ax[1, 0].scatter(stvarno, pred, s=12, alpha=0.4); ax[1, 0].plot([1, m], [1, m], "r--")
ax[1, 0].set_xscale("log"); ax[1, 0].set_yscale("log")
ax[1, 0].set_title("Naselje: stvarno vs predvidjeno"); ax[1, 0].set_xlabel("stvarno"); ax[1, 0].set_ylabel("predvidjeno")
mm = max(po_opstini.stvarno.max(), po_opstini.pred.max(), 1)
ax[1, 1].scatter(po_opstini.stvarno, po_opstini.pred, s=35); ax[1, 1].plot([1, mm], [1, mm], "r--")
ax[1, 1].set_title("Agregacija po opstini (R2 %.2f)" % metrike["agg_opstina_r2"])
ax[1, 1].set_xlabel("stvarno"); ax[1, 1].set_ylabel("predvidjeno")
plt.tight_layout()
mlflow.log_figure(fig, "evaluacija.png")
plt.show()

mlflow.pytorch.log_model(model, name="model")
mlflow.end_run()